# Import Required Libraries

In [ ]:
import numpy as np
import torch
from transformers import OPTForCausalLM
import torch.nn as nn
import torch.nn.functional as F
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# from torch.cuda.amp import autocast
from torch.amp import autocast,GradScaler
from typing import Optional, Tuple
import copy
import pickle
import random

# Define GPU Memory Monitoring Function


In [ ]:
def check_mem():
    if torch.cuda.is_available():
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        reserved_memory = torch.cuda.memory_reserved(0) / 1e9
        allocated_memory = torch.cuda.memory_allocated(0) / 1e9
        free_memory = reserved_memory - allocated_memory

        print(f"Total Memory: {total_memory:.2f} GB")
        print(f"Reserved Memory: {reserved_memory:.2f} GB")
        print(f"Allocated Memory: {allocated_memory:.2f} GB")
        print(f"Free (Unallocated) Memory: {free_memory:.2f} GB")
    else:
        print("CUDA is not available.")


# Define ArcFace Loss Function


In [ ]:
class ArcFaceLoss(nn.Module):
    def __init__(self, s=64.0, m=0.5, eps=1e-7):
        super(ArcFaceLoss, self).__init__()
        # self.in_features = in_features
        # self.out_features = out_features
        self.s = s
        self.m = m
        self.eps = eps
        # self.weight = nn.Parameter(torch.FloatTensor(out_features,in_features))
        # nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.threshold = math.cos(math.pi-m)
        self.mm = math.sin(math.pi - m) * m

    # def forward(self, input, label):
    def forward(self, cosine, label):

        # cosine = F.linear(F.normalize(input), F.normalize(self.weight2))
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1) + self.eps)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.threshold, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=cosine.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return F.cross_entropy(output, label)

# Load Pretrained Language Model

In [ ]:
name='facebook/opt-125m'
device='cuda:0'
# device='cpu'
# DEV='cuda:0'
DEV=device
# Load the pretrained model
model = OPTForCausalLM.from_pretrained(name,torch_dtype='auto')


# Install Datasets Library (Commented Out)

In [ ]:
# !pip install datasets

# Define Residual Model

This cell defines a residual model with two linear layers and a ReLU activation function. The model is designed to process hidden states and is initialized with the same data type as the pretrained model to ensure compatibility.

In [ ]:
class ResidualModel(nn.Module):
    def __init__(self,in_features, out_features,hid_features,dt):
        super(ResidualModel, self).__init__()
        self.w_proj1=nn.Linear(in_features=in_features,out_features=hid_features,bias=True)
        self.w_proj2=nn.Linear(in_features=hid_features,out_features=out_features,bias=True)

        # Convert the layers to the specified dtype
        self.w_proj1.weight.data = self.w_proj1.weight.data.to(dtype=dt)
        self.w_proj1.bias.data = self.w_proj1.bias.data.to(dtype=dt)

        self.w_proj2.weight.data = self.w_proj2.weight.data.to(dtype=dt)
        self.w_proj2.bias.data = self.w_proj2.bias.data.to(dtype=dt)

    def forward(self, x):
        x=self.w_proj1(x)
        x=self.w_proj2(torch.nn.functional.relu(x))
        return x

# Install Datasets Library

This cell installs the `datasets` library, which is required to load the C4 dataset for training and evaluation. The installation is executed to ensure the environment is properly configured.

In [ ]:
!pip install datasets

# Initialize Residual Model

This cell initializes the residual model with input and output dimensions matching the hidden size of the OPT-125M model (768) and a hidden layer size of 3072. The model is set to the same data type as the pretrained model.

In [ ]:
resmod=ResidualModel(768,768,3072,model.dtype)

# Load Pretrained Residual Model (Commented Out)


In [ ]:
# with open('/kaggle/input/modres/arcface_model14.pkl', 'rb') as f:
#     arcmod = pickle.load(f)


# Move Residual Model to Device

In [ ]:
print(next(resmod.parameters()).device)
resmod=resmod.to(DEV)
print(next(resmod.parameters()).device)

# Set Up Optimizer

In [ ]:
optimizer = torch.optim.Adam(list(resmod.parameters()) , lr=0.01)



# Load C4 Dataset


In [ ]:
from datasets import load_dataset
traindata3 = load_dataset('allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'})

# Define Tokenizer Function

Defines a function to load the tokenizer for the OPT-125M model, ensuring compatibility with the pretrained model for text tokenization.

In [ ]:
from transformers import AutoTokenizer
def get_tokenizer(model):
    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    return tokenizer

# Prepare Dataset for Tokenization

Selects a subset of the C4 dataset (first 1671 samples) and splits it into training (70%) and testing (30%) sets. It also loads the tokenizer for the OPT-125M model.

In [ ]:
orig_data=traindata3['train']['text'][:1671]
num=0.7*len(orig_data)
tokenizer=get_tokenizer('facebook/opt-125m')

# Tokenize Training Data

In [ ]:
trainenc=tokenizer("\n".join(orig_data[:int(num)]), return_tensors='pt')
print('train data')

# Tokenize Testing Data


In [ ]:
testenc=tokenizer("\n".join(orig_data[int(num):]), return_tensors='pt')
print('test_data')

# Data Loader Functions (Commented Out)

In [ ]:
# import random
# import numpy as np
# import torch
# from datasets import load_dataset
# from transformers import AutoTokenizer, LlamaTokenizer


# def set_seed(seed):
#     np.random.seed(seed)
#     torch.random.manual_seed(seed)

# def get_tokenizer(model):
#     # if "llama" in model.lower():
#     #     tokenizer = LlamaTokenizer.from_pretrained(model, use_fast=False)
#     #     # fix for transformer 4.28.0.dev0 compatibility
#     #     if tokenizer.bos_token_id != 1 or tokenizer.eos_token_id != 2:
#     #         try:
#     #             tokenizer.bos_token_id = 1
#     #             tokenizer.eos_token_id = 2
#     #         except AttributeError:
#     #             pass
#     # else:
#     tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
#     return tokenizer

# def get_wikitext2(nsamples, seed, seqlen, model, tokenizer):

#     # traindata = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
#     testdata = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')

#     # trainenc = tokenizer(" ".join(traindata['text']), return_tensors='pt')
#     testenc = tokenizer("\n\n".join(testdata['text']), return_tensors='pt')

#     random.seed(seed)
#     trainloader = []
#     # for _ in range(nsamples):
#     #     i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
#     #     j = i + seqlen
#     #     inp = trainenc.input_ids[:, i:j]
#     #     tar = inp.clone()
#     #     tar[:, :-1] = -100
#     #     trainloader.append((inp, tar))
#     return trainloader, testenc

# def get_ptb(nsamples, seed, seqlen, model, tokenizer):
#     traindata = load_dataset('ptb_text_only', 'penn_treebank', split='train')
#     testdata = load_dataset('ptb_text_only', 'penn_treebank', split='test')

#     trainenc = tokenizer(" ".join(traindata['sentence']), return_tensors='pt')
#     testenc = tokenizer(" ".join(testdata['sentence']), return_tensors='pt')

#     random.seed(seed)
#     trainloader = []
#     for _ in range(nsamples):
#         i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
#         j = i + seqlen
#         inp = trainenc.input_ids[:, i:j]
#         tar = inp.clone()
#         tar[:, :-1] = -100
#         trainloader.append((inp, tar))
#     return trainloader, testenc

# def get_c4(nsamples, seed, seqlen, model, tokenizer):
#     traindata = load_dataset(
#         'allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, split='train'
#     )
#     valdata = load_dataset(
#         'allenai/c4', data_files={'validation': 'en/c4-validation.00000-of-00008.json.gz'}, split='validation'
#     )

#     random.seed(seed)
#     trainloader = []
#     for _ in range(nsamples):
#         while True:
#             i = random.randint(0, len(traindata) - 1)
#             trainenc = tokenizer(traindata[i]['text'], return_tensors='pt')
#             if trainenc.input_ids.shape[1] > seqlen:
#                 break
#         i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
#         j = i + seqlen
#         inp = trainenc.input_ids[:, i:j]
#         tar = inp.clone()
#         tar[:, :-1] = -100
#         trainloader.append((inp, tar))

#     valenc = tokenizer(' '.join(valdata[:1100]['text']), return_tensors='pt')
#     valenc = valenc.input_ids[:, :(256 * seqlen)]

#     class TokenizerWrapper:
#         def __init__(self, input_ids):
#             self.input_ids = input_ids
#     valenc = TokenizerWrapper(valenc)

#     return trainloader, valenc

# def get_loaders(name, nsamples=128, seed=0, seqlen=2048, model=''):
#     tokenizer = get_tokenizer(model)
#     if 'wikitext2' in name:
#         return get_wikitext2(nsamples, seed, seqlen, model, tokenizer)
#     if 'ptb' in name:
#         return get_ptb(nsamples, seed, seqlen, model, tokenizer)
#     if 'c4' in name:
#         return get_c4(nsamples, seed, seqlen, model, tokenizer)


# Load WikiText Dataset (Commented Out)


In [ ]:
# for dataset in ['wikitext2']:
#     dataloader, testloader = get_loaders(
#         dataset, seed=0, model=name, seqlen=2048
#     )
#     print(dataset)

# Define Training and Evaluation Function

This cell defines the `arc_train` function, which handles both training and evaluation of the model. It processes the test dataset in chunks, applies the residual model, computes losses (including ArcFace loss during training), and calculates perplexity.

In [ ]:
import torch
import gc
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

# def opt_eval(model, testenc, dev, dataset: str, log_wandb: bool = False):
def arc_train(model, testenc, dev, arcmod, optimizer, dataset: str, log_wandb: bool = False,train=True):
    print('Evaluating ...')

    mseq = 2048  # Model sequence length
    testenc = testenc.input_ids
    nsamples = testenc.numel() // mseq
    # nsamples=2

    use_cache = model.config.use_cache
    model.config.use_cache = False
    layers = model.model.decoder.layers  # Get model layers

    # Move necessary components to GPU
    model.model.decoder.embed_tokens = model.model.decoder.embed_tokens.to(dev)
    model.model.decoder.embed_positions = model.model.decoder.embed_positions.to(dev)

    if hasattr(model.model.decoder, 'project_out') and model.model.decoder.project_out:
        model.model.decoder.project_out = model.model.decoder.project_out.to(dev)
    if hasattr(model.model.decoder, 'project_in') and model.model.decoder.project_in:
        model.model.decoder.project_in = model.model.decoder.project_in.to(dev)

    # Create buffer to store activations
    dtype = next(model.parameters()).dtype
    inps = torch.zeros((nsamples, mseq, model.config.hidden_size), dtype=dtype)
    cache = {'i': 0, 'attention_mask': None}

    # Catcher to intercept first-layer activations
    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp  # Store first-layer hidden state
            cache['i'] += 1
            cache['attention_mask'] = kwargs['attention_mask']
            raise ValueError  # Stop execution

    original_layer=copy.deepcopy(layers[0])
    print(type(original_layer))

    layers[0] = Catcher(layers[0])  # Wrap first layer with Catcher
    # layers[0] = Catcher(original_layer)

    # Run first layer to capture activations
    for i in range(nsamples):
        batch = testenc[:, (i * mseq):((i + 1) * mseq)].to(dev)
        try:
          # with autocast():
          model(batch)
        except ValueError:
            pass  # Catcher stops execution after first layer
    # print(f"Before restoring: {type(layers[0])}")
    # layers[0] = original_layer
    layers[0]=layers[0].module
    del original_layer
    # print(f"After restoring: {type(layers[0])}")
    assert not isinstance(layers[0], Catcher), "Catcher is still present!"
    torch.cuda.empty_cache()  # Free memory

    # Process layers sequentially with minimal memory usage
    attention_mask = cache['attention_mask']
    # deleted_layers=[2,3,7]
    deleted_layers=[-1]
    # deleted_layers=random.sample(range(0,5 ), 3)

    with torch.no_grad():
        # Disable gradient tracking to save memory
        with autocast():
            for i in range(len(layers)):
                if i not in deleted_layers:

                  print(f"Processing Layer {i}")

                  layer = layers[i].to(dev)  # Load one layer to GPU

                  for j in range(nsamples):

                      inps[j] = layer(inps[j].to(dev).unsqueeze(0), attention_mask=attention_mask)[0].to(torch.device('cpu'))  # Process in-place

                  layers[i] = layer.cpu()  # Move layer back to CPU
                  del layer
                  torch.cuda.empty_cache()  # Free up GPU memory


    nlls = []



    for i in range(nsamples):
      with torch.no_grad():
        if i%10==0:

          inps=inps.to('cpu')
          testenc = testenc.to('cpu')
          torch.cuda.empty_cache()
          if model.model.decoder.final_layer_norm is not None:
              model.model.decoder.final_layer_norm = model.model.decoder.final_layer_norm.to(dev)
          if model.model.decoder.project_out is not None:
              model.model.decoder.project_out = model.model.decoder.project_out.to(dev)

          # model.lm_head = model.lm_head.to(dev)
          model.lm_head = model.lm_head.to(dtype=torch.float32, device=dev)

          arcmod.w_proj1=arcmod.w_proj1.to(dtype=torch.float32,device=dev)
          arcmod.w_proj2=arcmod.w_proj2.to(dtype=torch.float32,device=dev)
          testenc = testenc.to(device=dev)
          inps=inps.to(dtype=torch.float32)


        hidden_states = inps[i].unsqueeze(0).to(dev)

        with autocast():
            if model.model.decoder.final_layer_norm is not None:
                hidden_states = model.model.decoder.final_layer_norm(hidden_states)

        # hidden_states = hidden_states.to(torch.float16)

        if model.model.decoder.project_out is not None:
            hidden_states = model.model.decoder.project_out(hidden_states)

      # with autocast():
      if train:
          optimizer.zero_grad()
          # hidden_states=hidden_states.to(torch.float32)

          temp=resmod(hidden_states)
          hidden_states=hidden_states+temp


          lm_logits = model.lm_head(hidden_states)



          lm_logits2 = nn.functional.linear(nn.functional.normalize(hidden_states), nn.functional.normalize(model.model.decoder.embed_tokens.weight))




          del hidden_states

          shift_logits2 = lm_logits2[:, :-1, :].contiguous()
          shift_logits = lm_logits[:, :-1, :].contiguous()
          shift_labels = testenc[:, (i * mseq):((i + 1) * mseq)][:, 1:]



          loss_fct=nn.CrossEntropyLoss()
          loss_arc=ArcFaceLoss(s=64)
          with torch.no_grad():
              loss_fct2=nn.CrossEntropyLoss()
              l=loss_fct2(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
              print(l)





          loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))+loss_arc(shift_logits2.view(-1, shift_logits2.size(-1)), shift_labels.view(-1))

      else:

          with torch.no_grad():

              temp=resmod(hidden_states)
              hidden_states=hidden_states+temp


              lm_logits = model.lm_head(hidden_states)

              # lm_logits2 = nn.functional.linear(nn.functional.normalize(hidden_states), nn.functional.normalize(model.model.decoder.embed_tokens.weight))


              del hidden_states

              # shift_logits2 = lm_logits2[:, :-1, :].contiguous()
              shift_logits = lm_logits[:, :-1, :].contiguous()
              shift_labels = testenc[:, (i * mseq):((i + 1) * mseq)][:, 1:]



              loss_fct=nn.CrossEntropyLoss()
              # loss_arc=ArcFaceLoss(s=64)



              l = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))






      neg_log_likelihood = l.float() * mseq

      neg_log_likelihood=neg_log_likelihood.to('cpu')

      print(f'sample {i} neg_log_likelihood {neg_log_likelihood}')
      print(l.item())
      nlls.append(neg_log_likelihood)



      if train:
        print('Is training...')
        loss.backward()
        optimizer.step()
        del loss

      del  lm_logits, shift_logits, shift_labels,l

      gc.collect()






    x1=torch.stack(nlls).sum()
    x2=(nsamples * mseq)


    ppl = torch.exp( x1/x2 )
    print(f"Perplexity: {ppl.item():.3f}")
    torch.cuda.empty_cache()

    if log_wandb:
        wandb.log({f'{dataset}/perplexity': ppl.item()})

    # Restore model's original config
    model.config.use_cache = use_cache



# Evaluate Model

This cell sets the model to evaluation mode and runs the `arc_train` function in evaluation mode (train=False) for one epoch. It processes the test dataset and computes perplexity to assess model performance.

In [ ]:

# print(dataset)
model.eval()
# opt_eval(model, testloader, DEV, dataset)
DEV = torch.device('cuda:0')
# DEV = torch.device('cpu')
testloader=testenc
dataset='aaaaa'
epochs=1
for i in range(epochs):
    print(i)
    print()
    arc_train(model, testloader, DEV,arcmod=resmod,optimizer=optimizer, dataset=dataset,train=False)

# Save Model Checkpoint

Moves the residual model to the CPU and saves it to a pickle file (`arcface_model16.pkl`) to preserve the trained parameters for future use.

In [ ]:
resmod=resmod.to('cpu')
with open('arcface_model1.pkl', 'wb') as f:
    pickle.dump(resmod, f)